**1.Importing libraries**

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**ARAI = Automotive Research Association of India,
LFP = Lithium Iron Phosphate
NMC = Nickel Manganese Cobalt**

In [47]:
ev_list = [
    {"make": "Tata", "model": "Nexon EV", "battery_kwh": 45.0, "arai_range_km": 489, "chemistry": "LFP"},
    {"make": "Tata", "model": "Punch EV", "battery_kwh": 40.0, "arai_range_km": 421, "chemistry": "LFP"},
    {"make": "MG", "model": "Comet EV", "battery_kwh": 17.3, "arai_range_km": 230, "chemistry": "LFP"},
    {"make": "Mahindra", "model": "XUV400", "battery_kwh": 39.4, "arai_range_km": 456, "chemistry": "NMC"},
    {"make": "Ather", "model": "450X", "battery_kwh": 3.7, "arai_range_km": 150, "chemistry": "NMC"},
    {"make": "Ola", "model": "S1 Pro", "battery_kwh": 4.0, "arai_range_km": 195, "chemistry": "NMC"}
]

**3. Converting into datafram**

In [48]:
df_evs = pd.DataFrame(ev_list)

In [49]:
np.random.seed(42)

# 5,000 random trips generate karenge humare vehicles se
total_trips = 5000
random_indices = np.random.choice(len(df_evs), size=total_trips)
df = df_evs.iloc[random_indices].reset_index(drop=True)

# Columns ke names readable aur clear kar rahe hain
df = df.rename(
    columns={
        "make": "Vehicle_Manufacturer",
        "model": "Vehicle_Model",
        "battery_kwh": "Battery_Capacity_kWh",
        "arai_range_km": "Claimed_ARAI_Range_km",
        "chemistry": "Battery_Chemistry",
    }
)

# Real driving conditions simulation
df["Initial_Battery_Percentage"] = np.random.uniform(20.0, 100.0, size=total_trips).round(1)
df["Outside_Temperature_Celsius"] = np.random.uniform(15.0, 48.0, size=total_trips).round(1)
df["Air_Conditioning_Status"] = np.random.choice([1, 0], size=total_trips, p=[0.7, 0.3])  # 1 = ON, 0 = OFF

''' p = [0.7,0.3] p = probability
| Value | Meaning | Probability |
| ----- | ------- | ----------- |
| 1     | AC ON   | 0.7 = 70%   |
| 0     | AC OFF  | 0.3 = 30%   |'''

df["Average_Speed_kmh"] = np.random.uniform(15.0, 85.0, size=total_trips).round(1)
df["Battery_Health_Percentage"] = np.random.uniform( 80.0, 100.0, size=total_trips).round(1)

# Physics calculation for Real-World Range

usable_energy_kwh = (df["Battery_Capacity_kWh"]* (df["Initial_Battery_Percentage"] / 100.0)* (df["Battery_Health_Percentage"] / 100.0))
heat_factor = np.maximum(0, df["Outside_Temperature_Celsius"] - 25.0)
ac_drain_kwh_per_km = df["Air_Conditioning_Status"] * (0.012 + 0.0008 * heat_factor)

#Base Energy Consumption base_km_per_km
base_kwh_per_km = df["Battery_Capacity_kWh"] / df["Claimed_ARAI_Range_km"] 

'''Battery = 40 kWh
ARAI Range = 400 km
Car approximately 1 km ke liye 0.1 kWh energy consume karti hai.'''

speed_penalty = np.where(df["Average_Speed_kmh"] > 60,1.0 + (df["Average_Speed_kmh"] - 60) * 0.01,1.0,)
'''np.where() basically:
    IF condition TRUE → first value
    IF condition FALSE → second value'''

#Total Energy Consumption per KM
total_kwh_per_km = (base_kwh_per_km * speed_penalty) + ac_drain_kwh_per_km

# Target Column 1: Actual Real Range
df["Actual_Real_Range_km"] = (usable_energy_kwh / total_kwh_per_km).round(1)

# Target Column 2: Battery Thermal Overheat Risk
thermal_score = (
    (df["Outside_Temperature_Celsius"] * 0.5)
    + (df["Average_Speed_kmh"] * 0.3)
    + (df["Air_Conditioning_Status"] * 10.0)
    + np.where(df["Battery_Chemistry"] == "NMC", 15.0, 0.0)
)

df["Battery_Thermal_Risk"] = np.where(
    thermal_score > 55, "High", np.where(thermal_score > 40, "Medium", "Low")
)

# Table details print karna
print("Dataset Shape:", df.shape)
df.sample(10)

Dataset Shape: (5000, 12)


,Vehicle_Manufacturer,Vehicle_Model,Battery_Capacity_kWh,Claimed_ARAI_Range_km,Battery_Chemistry,Initial_Battery_Percentage,Outside_Temperature_Celsius,Air_Conditioning_Status,Average_Speed_kmh,Battery_Health_Percentage,Actual_Real_Range_km,Battery_Thermal_Risk
244,Mahindra,XUV400,39.4,456,NMC,40.5,19.5,1,26.2,84.7,137.3,Medium
3585,Tata,Nexon EV,45.0,489,LFP,35.3,29.1,1,70.3,92.0,125.1,Medium
2327,Ola,S1 Pro,4.0,195,NMC,35.4,20.7,1,48.0,94.6,41.2,Medium
292,Mahindra,XUV400,39.4,456,NMC,52.5,30.3,1,68.3,92.4,174.0,High
1327,Ola,S1 Pro,4.0,195,NMC,88.7,21.3,1,70.9,88.6,90.5,High
3528,MG,Comet EV,17.3,230,LFP,58.6,45.4,1,15.2,85.6,83.8,Low
4983,Ather,450X,3.7,150,NMC,79.4,25.5,1,20.1,81.9,64.9,Medium
3780,Mahindra,XUV400,39.4,456,NMC,69.9,19.9,1,28.9,89.8,251.3,Medium
1282,Ather,450X,3.7,150,NMC,80.8,27.5,1,69.3,92.4,67.4,High
4410,Mahindra,XUV400,39.4,456,NMC,71.6,39.9,1,66.5,96.4,234.6,High


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Vehicle_Manufacturer         5000 non-null   object 
 1   Vehicle_Model                5000 non-null   object 
 2   Battery_Capacity_kWh         5000 non-null   float64
 3   Claimed_ARAI_Range_km        5000 non-null   int64  
 4   Battery_Chemistry            5000 non-null   object 
 5   Initial_Battery_Percentage   5000 non-null   float64
 6   Outside_Temperature_Celsius  5000 non-null   float64
 7   Air_Conditioning_Status      5000 non-null   int64  
 8   Average_Speed_kmh            5000 non-null   float64
 9   Battery_Health_Percentage    5000 non-null   float64
 10  Actual_Real_Range_km         5000 non-null   float64
 11  Battery_Thermal_Risk         5000 non-null   object 
dtypes: float64(6), int64(2), object(4)
memory usage: 468.9+ KB


In [51]:
df.describe()

,Battery_Capacity_kWh,Claimed_ARAI_Range_km,Initial_Battery_Percentage,Outside_Temperature_Celsius,Air_Conditioning_Status,Average_Speed_kmh,Battery_Health_Percentage,Actual_Real_Range_km
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,24.985760,324.420000,59.365860,31.454640,0.695400,50.895200,89.901300,142.085920
std,17.322353,136.060325,23.008272,9.499062,0.460284,20.263111,5.726662,96.089706
min,3.700000,150.000000,20.000000,15.000000,0.000000,15.000000,80.000000,10.800000
25%,4.000000,195.000000,39.400000,23.300000,0.000000,33.500000,85.000000,64.600000
50%,39.400000,421.000000,58.900000,31.350000,1.000000,51.100000,89.900000,114.500000
75%,40.000000,456.000000,79.200000,39.700000,1.000000,68.325000,94.700000,207.125000
max,45.000000,489.000000,100.000000,48.000000,1.000000,85.000000,100.000000,462.300000


**CATEGORICAL DESCRIPTION**

In [52]:
df.select_dtypes(include=['object']).describe()

,Vehicle_Manufacturer,Vehicle_Model,Battery_Chemistry,Battery_Thermal_Risk
count,5000,5000,5000,5000
unique,5,6,2,3
top,Tata,Nexon EV,NMC,Medium
freq,1673,870,2515,2318


In [53]:
df.isnull().sum()

Vehicle_Manufacturer           0
Vehicle_Model                  0
Battery_Capacity_kWh           0
Claimed_ARAI_Range_km          0
Battery_Chemistry              0
Initial_Battery_Percentage     0
Outside_Temperature_Celsius    0
Air_Conditioning_Status        0
Average_Speed_kmh              0
Battery_Health_Percentage      0
Actual_Real_Range_km           0
Battery_Thermal_Risk           0
dtype: int64

### 📌 Feature Selection Decision: Dropping Metadata Columns

We are dropping `Vehicle_Manufacturer` and `Vehicle_Model` from our input features ($X$) due to the following engineering reasons:

1. **Elimination of Brand Bias:** 
   Battery thermodynamics and range prediction depend purely on physical parameters—such as **Battery Capacity (kWh)**, **Chemistry (LFP/NMC)**, **Speed**, **Ambient Temperature**, and **AC Load**—not on the manufacturer's brand badge.

2. **Handling Unseen Vehicles (Zero-Shot Generalization):** 
   If a user inputs a brand-new EV in production (e.g., *Skoda Kylaq EV* or *Hyundai Creta EV*), a model trained on specific brand names would crash or fail due to unseen categories (`OneHotEncoder` Out-of-Vocabulary error). By focusing strictly on physical features, the model works universally for any current or future EV.

3. **Preventing Model Overfitting:** 
   Including static model names causes machine learning algorithms to memorize brand-specific averages instead of learning the underlying physics and environmental impact on battery drain.

4. **Production Microservice Optimization:** 
   Dropping metadata simplifies the REST API request schema, reduces pre-processing latency, and prevents coupling our model pipeline to a fixed vehicle lookup database.

In [54]:
df.shape

(5000, 12)

In [55]:
X = df.drop(columns=[
    'Vehicle_Manufacturer', 
    'Vehicle_Model', 
    'Actual_Real_Range_km', 
    'Battery_Thermal_Risk'
])

# 2. Select Targets (y)
y_range = df['Actual_Real_Range_km']      # For Regression Model
y_risk = df['Battery_Thermal_Risk']       # For Classification Model

print("Input Features (X) Shape:", X.shape)
print("\nInput Columns Used for ML Training:")
print(X.columns.tolist())

Input Features (X) Shape: (5000, 8)

Input Columns Used for ML Training:
['Battery_Capacity_kWh', 'Claimed_ARAI_Range_km', 'Battery_Chemistry', 'Initial_Battery_Percentage', 'Outside_Temperature_Celsius', 'Air_Conditioning_Status', 'Average_Speed_kmh', 'Battery_Health_Percentage']


In [56]:
df.shape

(5000, 12)

In [57]:
X.head()

,Battery_Capacity_kWh,Claimed_ARAI_Range_km,Battery_Chemistry,Initial_Battery_Percentage,Outside_Temperature_Celsius,Air_Conditioning_Status,Average_Speed_kmh,Battery_Health_Percentage
0,39.4,456,NMC,51.2,17.3,1,41.0,91.4
1,3.7,150,NMC,63.3,21.2,1,47.3,96.9
2,17.3,230,LFP,97.4,40.5,1,73.4,98.4
3,3.7,150,NMC,25.3,42.8,1,61.2,89.7
4,3.7,150,NMC,71.9,35.4,0,19.1,92.8


In [58]:
y_range.head(),y_risk.head()

(0    187.4
 1     61.9
 2    151.1
 3     16.4
 4    100.1
 Name: Actual_Real_Range_km, dtype: float64,
 0    Medium
 1    Medium
 2    Medium
 3      High
 4       Low
 Name: Battery_Thermal_Risk, dtype: object)

# **Train-Test Split & Preprocessing Pipeline**

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [60]:
# 1. Train-Test Split (80% Train, 20% Test)
X_train,X_test,y_range_train,y_range_test,y_risk_train,y_risk_test =  train_test_split(
    X,y_range,y_risk,test_size=0.20,random_state=42)

# 2. Grouping Columns by Data Type
numeric_cols = [
    'Battery_Capacity_kWh', 
    'Claimed_ARAI_Range_km', 
    'Initial_Battery_Percentage', 
    'Outside_Temperature_Celsius', 
    'Air_Conditioning_Status', 
    'Average_Speed_kmh', 
    'Battery_Health_Percentage'
]
categorical_cols = ['Battery_Chemistry']

In [61]:
#3.Preprocessing Transformer engine
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numeric_cols),
        ('cat',OneHotEncoder(drop='first',sparse_output=False),categorical_cols)
    ]
)

In [62]:
# 4. Fit and Transform Training Data
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

In [63]:
print("✅ Data Preprocessing Complete!")
print(f"X_train Shape: {X_train.shape} -> Transformed: {X_train_scaled.shape}")
print(f"X_test Shape: {X_test.shape} -> Transformed: {X_test_scaled.shape}")

✅ Data Preprocessing Complete!
X_train Shape: (4000, 8) -> Transformed: (4000, 8)
X_test Shape: (1000, 8) -> Transformed: (1000, 8)


# **Step 4: Model Training (Stage 4)**

**Model 1 (RandomForestRegressor): Real Range Predict karne ke liye ($R^2$ Score check karenge).Model 2 (RandomForestClassifier): Thermal Overheat Risk Predict karne ke liye (Accuracy % check karenge).**

In [64]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, classification_report

# ==========================================
# 1. MODEL 1: REAL RANGE PREDICTOR (REGRESSION)
# ==========================================
range_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train Model 1
range_pipeline.fit(X_train, y_range_train)

# Predict & Evaluate
y_range_pred = range_pipeline.predict(X_test)
r2 = r2_score(y_range_test, y_range_pred)
mae = mean_absolute_error(y_range_test, y_range_pred)

print("--- 🎯 MODEL 1: REAL RANGE PREDICTOR PERFORMANCE ---")
print(f"R² Score: {r2:.4f} (1.0 is Perfect)")
print(f"Mean Absolute Error (MAE): ±{mae:.2f} km")


# ==========================================
# 2. MODEL 2: THERMAL RISK CLASSIFIER (CLASSIFICATION)
# ==========================================
risk_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train Model 2
risk_pipeline.fit(X_train, y_risk_train)

# Predict & Evaluate
y_risk_pred = risk_pipeline.predict(X_test)
accuracy = accuracy_score(y_risk_test, y_risk_pred)

print("\n--- ⚠️ MODEL 2: THERMAL RISK CLASSIFIER PERFORMANCE ---")
print(f"Accuracy Score: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_risk_test, y_risk_pred))

--- 🎯 MODEL 1: REAL RANGE PREDICTOR PERFORMANCE ---
R² Score: 0.9941 (1.0 is Perfect)
Mean Absolute Error (MAE): ±5.04 km

--- ⚠️ MODEL 2: THERMAL RISK CLASSIFIER PERFORMANCE ---
Accuracy Score: 96.20%

Classification Report:
               precision    recall  f1-score   support

        High       0.99      0.91      0.95       227
         Low       0.98      0.97      0.98       315
      Medium       0.94      0.98      0.96       458

    accuracy                           0.96      1000
   macro avg       0.97      0.95      0.96      1000
weighted avg       0.96      0.96      0.96      1000



In [65]:
import pandas as pd

# 1. Simulating a raw API request payload for a new vehicle trip
sample_api_input = pd.DataFrame([{
    'Battery_Capacity_kWh': 40.0,
    'Claimed_ARAI_Range_km': 421.0,
    'Battery_Chemistry': 'LFP',
    'Initial_Battery_Percentage': 75.0,
    'Outside_Temperature_Celsius': 42.0,  # Extreme heat
    'Air_Conditioning_Status': 1,         # AC ON
    'Average_Speed_kmh': 35.0,            # Traffic speed
    'Battery_Health_Percentage': 92.0
}])

# 2. Generate Predictions directly using raw pipeline
predicted_range = range_pipeline.predict(sample_api_input)[0]
predicted_risk = risk_pipeline.predict(sample_api_input)[0]

print("--- 🚗 REAL-TIME API PREDICTION TEST ---")
print(f"Input Trip Conditions: 42°C Heat, AC ON, 75% Battery")
print(f"Predicted Real Range: {predicted_range:.1f} km (vs 421 km ARAI)")
print(f"Predicted Thermal Overheat Risk: {predicted_risk}")

--- 🚗 REAL-TIME API PREDICTION TEST ---
Input Trip Conditions: 42°C Heat, AC ON, 75% Battery
Predicted Real Range: 233.4 km (vs 421 km ARAI)
Predicted Thermal Overheat Risk: Medium


In [66]:
import joblib

# Re-saving pipelines with compression (compress=3)
joblib.dump(range_pipeline, 'ev_range_model_pipeline.joblib', compress=3)
joblib.dump(risk_pipeline, 'ev_thermal_risk_pipeline.joblib', compress=3)

print("✅ Compressed pipelines saved successfully!")
print("Saved files: 'ev_range_model_pipeline.joblib' & 'ev_thermal_risk_pipeline.joblib'")

✅ Compressed pipelines saved successfully!
Saved files: 'ev_range_model_pipeline.joblib' & 'ev_thermal_risk_pipeline.joblib'


In [67]:
!pip install fastapi uvicorn pydantic

In [68]:
!pip install xgboost

In [73]:
import time
import pandas as pd
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Models initialization
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": range_pipeline.named_steps['regressor'], # Pre-trained RF
    "XGBoost Regressor": XGBRegressor(n_estimators=100, random_state=42)
}

results = []

# 2. Extract transformed features using preprocessor
X_train_prep = range_pipeline.named_steps['preprocessor'].transform(X_train)
X_test_prep = range_pipeline.named_steps['preprocessor'].transform(X_test)

# 3. Benchmarking loop
for name, model in models.items():
    start_time = time.time()
    
    if name == "Random Forest Regressor":
        # Direct prediction via range_pipeline
        y_pred = range_pipeline.predict(X_test)
    else:
        # Fit model on preprocessed training data and exact target variable y_range_train
        model.fit(X_train_prep, y_range_train)
        y_pred = model.predict(X_test_prep)
        
    latency = (time.time() - start_time) * 1000  # in milliseconds
    
    # Evaluation using exact test target variable y_range_test
    r2 = r2_score(y_range_test, y_pred)
    mae = mean_absolute_error(y_range_test, y_pred)
    
    results.append({
        "Model": name,
        "R2 Score": round(r2, 4),
        "MAE (km)": round(mae, 2),
        "Inference Time (ms)": round(latency, 2)
    })

# 4. Display Comparison Table
benchmark_df = pd.DataFrame(results)
print("--- 📊 ALGORITHM BENCHMARKING REPORT ---")
display(benchmark_df)

--- 📊 ALGORITHM BENCHMARKING REPORT ---


,Model,R2 Score,MAE (km),Inference Time (ms)
0,Linear Regression,0.8993,24.42,29.31
1,Random Forest Regressor,0.9941,5.04,28.77
2,XGBoost Regressor,0.9972,3.60,1535.66
